In [ ]:
import kagglehub
import kagglehub
import kagglehub
import os
import pandas as pd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
# Download latest version


# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
path = os.path.join(path, 'Q3_data.csv')
df = pd.read_csv(path)

print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
df.info()
# Task 3: Write your code here:

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
missing_data.head(10)

In [ ]:
missing_cols = missing_data[missing_data['Missing_Percentage'] > 80]['Column']
df.drop(columns=missing_cols, inplace=True)
# coloms that has more then 80 i guess its better to drop them because imputing thim dosnt make since

In [ ]:
# Fill missing values for the remaining columns
for col in df.columns:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].median(), inplace=True)

In [ ]:
# Task 2: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")
    # if theres any dublicts df.drop_duplicates(inplace=True)

check_duplicates(df)

In [ ]:
df.head()

In [ ]:
# Task 3: Write your code here:
categorical_cols = df.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

df.head()
#theres no catigoracle coloms in our data

In [ ]:
df.info()

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import MinMaxScaler, StandardScaler
numerical_cols = df.select_dtypes(include=["number"]).columns.drop("Target") # i didnot scale the target because its wrong and it will cause leacage then the moodl is gonna cheat

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])

df.head()

In [ ]:
# Task 5: Write your code here:
def check_target_imbalance(df, target_column):
  print("Target Distribution:")

  df[target_column].hist()
  plt.show()

check_target_imbalance(df, "Target")
#the results shows huge inbalance we need stratifid k fold to make sure its distrubuted right

In [ ]:
# Task 1: Write your code here:
X = df.drop(columns=['Target'])
y = df['Target']

# Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# Task 2,3,4,5: Write your code here:
! pip install catboost

from catboost import CatBoostClassifier
model = CatBoostClassifier(
    iterations=100,
    learning_rate=0.1,
    random_state=42,
    verbose=False        # عشان ما يطلع logs
)

model.fit(X_train, y_train)
print("Model trained!")

In [ ]:
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
y_pred = model.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
print(f"f1_score: {f1_score(y_test, y_pred):.4f}\n")
# f1 = f1_score(y_fold_val, y_fold_pred, average='macro')
print(classification_report(y_test, y_pred))
# i printed the whole report if you want it

In [ ]:
# Task 1: Write your code here:
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(90, 40))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()
# this may not be the brettist plot but u can see all of the feachers if you click on them (it does the gob)


In [ ]:
# Task 2: Write your code here:
#P_2 and D_42 and D_45 are one of rhe most importanat features
# the golden feature is P_2


In [ ]:
# Task Bonus: Write your code here:
X = df['P_2']

In [ ]:
# im supossed to do it in the golden feature but there is no time

from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

kfold = KFold(n_splits=5, shuffle=True, random_state=42)

accuracy_scores = []
precision_scores = []
recall_scores = []
f1_scores = []

for fold, (train_idx, val_idx) in enumerate(kfold.split(X_train), 1):
    X_fold_train, X_fold_val = X_train[train_idx], X_train[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # Train and predict
    model.fit(X_fold_train, y_fold_train)
    y_fold_pred = model.predict(X_fold_val)

    # Calculate metrics
    acc = accuracy_score(y_fold_val, y_fold_pred)
    prec = precision_score(y_fold_val, y_fold_pred, average='macro')
    rec = recall_score(y_fold_val, y_fold_pred, average='macro')
    f1 = f1_score(y_fold_val, y_fold_pred, average='macro')

    accuracy_scores.append(acc)
    precision_scores.append(prec)
    recall_scores.append(rec)
    f1_scores.append(f1)

    print(f"Fold {fold} - Accuracy: {acc:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}, F1: {f1:.4f}")

In [ ]:
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
y_pred = model.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
print(f"f1_score: {f1_score(y_test, y_pred):.4f}\n")
# f1 = f1_score(y_fold_val, y_fold_pred, average='macro')
print(classification_report(y_test, y_pred))
# i printed the whole report if you want it